# Trade-off Curves (Accuracy vs AIR)

This notebook shows how changing the **cut score** changes both:
- business utility (accuracy)
- fairness (AIR)

In applied work, the cut score is a governance decision: it should be chosen transparently and documented.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../02_standards_aligned_pipeline')
import scoring_function as sf

rng = np.random.default_rng(7)
n = 2500
df = pd.DataFrame({
    'Protected_Group': rng.integers(0, 2, size=n),
    'High_Performer': rng.binomial(1, 0.28, size=n),
    'Retained': rng.binomial(1, 0.60, size=n),
})
df['score'] = 1.2*df['High_Performer'] + 1.0*df['Retained'] + rng.normal(0, 0.7, size=n)

thresholds = np.quantile(df['score'], np.linspace(0.05, 0.95, 50))
tbl = sf.policy_table(df, 'score', 'Protected_Group', 'High_Performer', 'Retained', thresholds)

plt.figure()
plt.plot(tbl['AIR'], tbl['overall_accuracy'])
plt.xlabel('Adverse Impact Ratio (AIR)')
plt.ylabel('Overall accuracy score')
plt.title('Illustrative accuracy–AIR trade-off')
plt.grid(True)
plt.show()

tbl.sort_values('final_score', ascending=False).head(10)
